# Results section 5, first half, and panels a and b of Figure 4

## Setup

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings("ignore")

# the package root is the folder that holds Data/
def _root(start=None):
    here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd().resolve()
    for c in (here, *here.parents):
        if (c / "Data" / "conversation_turns.csv").exists():
            return c
    raise RuntimeError(f"cannot find the package root (no Data/conversation_turns.csv at or above {here})")

ROOT   = _root()
CLEAN  = ROOT / "Data"
RESDIR = ROOT / "Code" / "_output" / "results"; RESDIR.mkdir(parents=True, exist_ok=True)

### Helper functions

In [2]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

def glmm(formula, data, crossed, label=""):
    # crossed random intercepts, variational Bayes
    vc = {g: f"0 + C({g})" for g in crossed}
    # seed the variational fit
    np.random.seed(2026)
    m = BinomialBayesMixedGLM.from_formula(formula, vc, data).fit_vb()
    if label:
        print(f"  {label}")
    print(f"    {'term':<18}{'b':>9}{'SD':>8}{'95% CrI':>21}{'odds':>8}")
    for i, nm in enumerate(m.model.exog_names):
        lo, hi = m.fe_mean[i] - 1.96 * m.fe_sd[i], m.fe_mean[i] + 1.96 * m.fe_sd[i]
        print(f"    {nm:<18}{m.fe_mean[i]:>9.3f}{m.fe_sd[i]:>8.3f}"
              f"   [{lo:+.3f}, {hi:+.3f}]{np.exp(m.fe_mean[i]):>8.2f}")
    print("    random intercepts (SD, logit scale): "
          + "  ".join(f"{n} {np.exp(v):.2f}" for n, v in zip(m.model.vcp_names, m.vcp_mean)))
    return m

def robust_logit(formula, data, cluster):
    return smf.logit(formula, data=data).fit(disp=0, cov_type="cluster",
                                             cov_kwds={"groups": data[cluster]})

def paired_effect(x, y, unit_label, names=("first", "second")):
    d = np.asarray(y) - np.asarray(x)
    n = len(d); m, sd = d.mean(), d.std(ddof=1)
    se = sd / np.sqrt(n)
    tc = stats.t.ppf(0.975, n - 1)
    t = stats.ttest_rel(y, x)
    print(f"    {names[1]} - {names[0]} = {m:+.3f}   95% CI "
          f"[{m - tc*se:+.3f}, {m + tc*se:+.3f}]   "
          f"t({n-1}) = {t.statistic:+.2f}, p = {t.pvalue:.1e}")
    print(f"    Cohen's dz = {m / sd:+.2f}   ({unit_label}, n = {n})")
    return m, m / sd

## 1 · Words recalled in the two individual recalls

In [3]:
RECALL = pd.read_csv(CLEAN / "individual_recall.csv")
TURNS  = pd.read_csv(CLEAN / "conversation_turns.csv").sort_values(
    ["conversation", "turn_index"])
STIM   = pd.read_csv(CLEAN / "stimuli.csv")
ITEMS  = {int(i): int(l) for i, l in zip(STIM[~STIM.is_lure].item_id,
                                        STIM[~STIM.is_lure].list)}

def _sets(phase, kind="studied"):
    d = RECALL[(RECALL.phase == phase) & (RECALL.kind == kind)]
    return {str(p): set(g.item_id.dropna().astype(int)) for p, g in d.groupby("participant")}

PRE, POST = _sets("pre"), _sets("post")
OWN_C = {}                      # the private recall as item-id sets per category
for _p, _v in PRE.items():
    OWN_C[_p] = {}
    for _i in _v:
        OWN_C[_p].setdefault(ITEMS[_i], set()).add(_i)

# what each person said, and what they heard the other person say, across their conversations
SAID, HEARD, NET = {}, {}, {}
for cid, g in TURNS.groupby("conversation"):
    roles = list(dict.fromkeys(g.speaker_role))
    pid = dict(zip(g.speaker_role, g.participant.astype(str)))
    for p in pid.values():
        NET[p] = str(g.network.iloc[0])
    for _, r in g.iterrows():
        if r.kind != "studied" or pd.isna(r.item_id):
            continue
        w = int(r.item_id); sp = pid[r.speaker_role]
        other = [pid[x] for x in roles if x != r.speaker_role][0]
        SAID.setdefault(sp, set()).add(w)
        HEARD.setdefault(other, set()).add(w)

PP = pd.DataFrame([dict(person=p, n_pre=len(a), n_post=len(POST.get(p, set())))
                   for p, a in PRE.items()])
print("  studied words recalled, second recall against first")
_ = paired_effect(PP.n_pre, PP.n_post, "person as the unit", ("before", "after"))

  studied words recalled, second recall against first
    after - before = +8.694   95% CI [+7.663, +9.725]   t(169) = +16.65, p = 1.8e-37
    Cohen's dz = +1.28   (person as the unit, n = 170)


## 2 · Missed words recalled later, by exposure

In [4]:
rows = []
for p, pre in PRE.items():
    s, h = SAID.get(p, set()), HEARD.get(p, set())
    for w, lst in ITEMS.items():
        if w in pre:
            continue
        rows.append(dict(person=p, word=str(w), category=str(lst), network=NET.get(p, "?"),
                         exposure="said" if w in s else ("heard" if w in h else "never"),
                         acquired=int(w in POST.get(p, set()))))
ACQ = pd.DataFrame(rows)
_ = glmm("acquired ~ C(exposure, Treatment('never'))", ACQ,
         ["person", "word", "category", "network"],
         "crossed on person, word, category and network (odds: OR against 'never')")

  crossed on person, word, category and network (odds: OR against 'never')
    term                      b      SD              95% CrI    odds
    Intercept            -3.394   0.030   [-3.453, -3.335]    0.03
    C(exposure, Treatment('never'))[T.heard]    2.191   0.039   [+2.115, +2.267]    8.95
    C(exposure, Treatment('never'))[T.said]    4.513   0.065   [+4.386, +4.641]   91.24
    random intercepts (SD, logit scale): person 0.64  word 0.50  category 0.30  network 0.07


## 3 · Later recall of partner words, by patch continuation

In [5]:
def patch_blocks(cats):
    out, cur = [], []
    for i, c in enumerate(cats):
        if c is None:
            if cur:
                out.append(cur); cur = []
            continue
        if cur and cats[cur[-1]] == c:
            cur.append(i)
        else:
            if cur:
                out.append(cur)
            cur = [i]
    if cur:
        out.append(cur)
    return out

rows = []
for cid, g in TURNS.groupby("conversation"):
    cats = [None if pd.isna(v) else int(v) for v in g.list]
    ids = [None if pd.isna(v) else int(v) for v in g.item_id]
    spk = list(g.speaker_role)
    pid = dict(zip(g.speaker_role, g.participant.astype(str)))
    nth = dict(zip(g.speaker_role, g.nth_conversation_for_speaker))
    n = len(cats)
    for b in patch_blocks(cats):
        t = b[0]
        if ids[t] is None:                      # the opener has to be a studied word
            continue
        sy = spk[t]; hr = [x for x in pid if x != sy][0]
        # the hearer must hold the next turn; otherwise the pair is censored
        if t + 1 >= n or spk[t + 1] != hr:
            continue
        hearer = pid[hr]; w = ids[t]
        if w in PRE.get(hearer, set()):         # only words that were new to the hearer
            continue
        rows.append(dict(person=hearer, word=str(w), category=str(cats[t]),
                         conversation=cid, network=str(g.network.iloc[0]),
                         nth=int(nth[hr]), turn=t / max(n - 1, 1),
                         length=len(b), survived=int(len(b) > 1),
                         kept=int(w in POST.get(hearer, set()))))
HEARD_W = pd.DataFrame(rows)
HEARD_W["n_exp"] = HEARD_W.groupby(["person", "word"]).person.transform("size")
# a word heard as a patch opener more than once is kept at its earliest exposure
KEEP = (HEARD_W.sort_values(["person", "nth", "turn"])
        .drop_duplicates(["person", "word"]).copy())
KEEP["nthf"] = KEEP.nth.astype(str)

In [6]:
_ = glmm("kept ~ survived + turn + n_exp + C(nthf)", KEEP,
         ["person", "word", "category", "conversation", "network"],
         "crossed on the hearer, the word, the category, the conversation and the network")
# the same model with standard errors clustered on the participant (Supplementary Note 1)
mkp = robust_logit("kept ~ survived + turn + n_exp + C(nthf)", KEEP, "person")
print(f"    clustered on the participant: b = {mkp.params['survived']:+.3f}, "
      f"SE = {mkp.bse['survived']:.3f}, p = {mkp.pvalues['survived']:.1e}")
print("\n  bootstrap over the 17 networks (Supplementary Note 1)")
rng = np.random.default_rng(5); nets = KEEP.network.unique(); bt = []
for _ in range(2000):
    d = pd.concat([KEEP[KEEP.network == nn]
                   for nn in rng.choice(nets, len(nets), replace=True)])
    try:
        bt.append(smf.logit("kept ~ survived + turn + n_exp + C(nthf)", d)
                  .fit(disp=0).params["survived"])
    except Exception:
        pass
bt = np.array(bt)
print(f"    survived {bt.mean():+.3f}   95% CI [{np.percentile(bt, 2.5):+.3f}, "
      f"{np.percentile(bt, 97.5):+.3f}]")

  crossed on the hearer, the word, the category, the conversation and the network
    term                      b      SD              95% CrI    odds
    Intercept            -1.148   0.038   [-1.222, -1.074]    0.32
    C(nthf)[T.2]         -0.210   0.065   [-0.339, -0.082]    0.81
    C(nthf)[T.3]         -0.337   0.070   [-0.473, -0.200]    0.71
    survived              0.585   0.091   [+0.406, +0.764]    1.80
    turn                  0.138   0.067   [+0.007, +0.269]    1.15
    n_exp                 0.355   0.030   [+0.297, +0.413]    1.43
    random intercepts (SD, logit scale): person 0.61  word 0.56  category 0.31  conversation 0.11  network 0.09
    clustered on the participant: b = +0.645, SE = 0.100, p = 1.2e-10

  bootstrap over the 17 networks (Supplementary Note 1)
    survived +0.649   95% CI [+0.503, +0.805]


### 3.1 · Each of the three conversations (Supplementary Note 5.1)

In [7]:
_bn = []
for _k in (1, 2, 3):
    _d = KEEP[KEEP.nth == _k]
    for _sv in (0, 1):
        _pp = _d[_d.survived == _sv].groupby("person").kept.mean()
        _bn.append(dict(nth=_k, survived=_sv, per_person=_pp.mean(), per_sd=_pp.std(ddof=1),
                        n_people=len(_pp)))
print("  proportion recalled later, mean (SD) across participants")
print(pd.DataFrame(_bn).round(3).to_string(index=False))

  proportion recalled later, mean (SD) across participants
 nth  survived  per_person  per_sd  n_people
   1         0       0.406   0.277       157
   1         1       0.526   0.422       105
   2         0       0.340   0.265       162
   2         1       0.484   0.435       106
   3         0       0.312   0.271       160
   3         1       0.439   0.420       103


### 3.2 · Controlling for the offered and own stores

In [8]:
rows = []
for cid, g in TURNS.groupby("conversation"):
    cats = [None if pd.isna(v) else int(v) for v in g.list]
    ids = [None if pd.isna(v) else int(v) for v in g.item_id]
    spk = list(g.speaker_role)
    pid = dict(zip(g.speaker_role, g.participant.astype(str)))
    for b in patch_blocks(cats):
        t = b[0]
        if ids[t] is None:
            continue
        sy = spk[t]; hr = [x for x in pid if x != sy][0]
        me = pid[hr]; w = ids[t]; cat = cats[t]
        # the same censoring as KEEP: the hearer must hold the next turn
        if t + 1 >= len(cats) or spk[t + 1] != hr:
            continue
        if w in PRE.get(me, set()):
            continue
        off = len(OWN_C.get(me, {}).get(cat, set())
                  - {ids[u] for u in range(t + 1) if cats[u] == cat})
        prev = [u for u in range(t) if spk[u] == hr and cats[u] is not None]
        own = np.nan
        if prev and cats[prev[-1]] != cat:
            oc = cats[prev[-1]]
            own = len(OWN_C.get(me, {}).get(oc, set())
                      - {ids[u] for u in range(t) if cats[u] == oc})
        rows.append(dict(person=me, word=str(w), conversation=cid, t=t,
                         off_left=off, own_left=own))
# stores taken from the same (earliest) exposure as in KEEP
STORES = (pd.DataFrame(rows).sort_values(["person", "word", "conversation", "t"])
          .drop_duplicates(["person", "word", "conversation"]).drop(columns="t"))
KEEP2 = KEEP.merge(STORES, on=["person", "word", "conversation"], how="left")
# common sample: pairs where the competing patch is measurable
COMMON = KEEP2.dropna(subset=["own_left"]).copy()
print(f"  common sample: {len(COMMON)} pairs where the responder's own store is measurable\n")

F0 = "kept ~ survived + turn + n_exp + C(nthf)"
F2 = "kept ~ survived + off_left + own_left + turn + n_exp + C(nthf)"
print(f"  {'model':<26}{'b':>8}{'SE':>7}{'OR':>7}{'95% CI (OR)':>18}")
for f_, lab in ((F0, "without the stores"), (F2, "+ both stores")):
    m_ = robust_logit(f_, COMMON, "conversation")
    b, se = m_.params["survived"], m_.bse["survived"]
    print(f"  {lab:<26}{b:>8.3f}{se:>7.3f}{np.exp(b):>7.2f}"
          f"   [{np.exp(b - 1.96 * se):.2f}, {np.exp(b + 1.96 * se):.2f}]")

  common sample: 3302 pairs where the responder's own store is measurable

  model                            b     SE     OR       95% CI (OR)
  without the stores           0.648  0.091   1.91   [1.60, 2.29]
  + both stores                0.650  0.095   1.92   [1.59, 2.31]


## 4 · The data for panels a and b of Figure 4

In [9]:
rgb = np.random.default_rng(31)

def binned(df, col, y, cap=4, cluster="person", nboot=2000, minn=25):
    out = []
    d = df.assign(b=df[col].clip(upper=cap))
    units = d[cluster].unique()
    for k in sorted(d.b.unique()):
        sub = d[d.b == k]
        if len(sub) < minn:          # skip bins with fewer than minn rows
            continue
        # per-cluster sums and counts
        agg = sub.groupby(cluster)[y].agg(["sum", "size"]).reindex(units, fill_value=0)
        S_, N_ = agg["sum"].values, agg["size"].values
        draws = []
        for _ in range(nboot):
            ix = rgb.integers(0, len(units), len(units))
            if N_[ix].sum():
                draws.append(S_[ix].sum() / N_[ix].sum())
        out.append((k, sub[y].mean(), np.percentile(draws, 2.5),
                    np.percentile(draws, 97.5), len(sub)))
    return pd.DataFrame(out, columns=["b", "m", "lo", "hi", "n"])

In [10]:
MERGED = RESDIR / "merged"; MERGED.mkdir(exist_ok=True)

# ---- panel a · where the new words come from, one value per person ----------------------
_n = ACQ.groupby(["person", "exposure"]).size().unstack(fill_value=0)
_r = ACQ.groupby(["person", "exposure"]).acquired.mean().unstack()
PERPERSON = _r[(_n >= 5).all(axis=1)][["never", "heard", "said"]]
# one row per person, without the participant id
PERPERSON.to_csv(MERGED / "f5_exposure_per_person.csv", index=False)
print(f"  panel a: {len(PERPERSON)} people with at least five opportunities in all three cells")
print(PERPERSON.agg(["mean", "median", "std"]).round(3).to_string())

# ---- panel b · retention by patch continuation ------------------------------------------
_rows = []
for sv, lab in ((0, "ended"), (1, "continued")):
    g = binned(KEEP[KEEP.survived == sv].assign(one=0), "one", "kept", cap=0,
               cluster="conversation")
    _rows.append(dict(survived=sv, label=lab, rate=g.m[0], lo=g.lo[0], hi=g.hi[0], n=int(g.n[0])))
KEPT_BY_SURVIVAL = pd.DataFrame(_rows)
KEPT_BY_SURVIVAL.to_csv(MERGED / "f5_kept_by_survival.csv", index=False)
print("\n  panel b")
print(KEPT_BY_SURVIVAL.round(3).to_string(index=False))
print(f"\n  wrote results/merged/f5_exposure_per_person.csv and f5_kept_by_survival.csv")

  panel a: 133 people with at least five opportunities in all three cells
exposure  never  heard   said
mean      0.036  0.268  0.776
median    0.018  0.267  0.818
std       0.063  0.131  0.192

  panel b
 survived     label  rate    lo    hi    n
        0     ended 0.335 0.315 0.356 2992
        1 continued 0.484 0.443 0.524  570

  wrote results/merged/f5_exposure_per_person.csv and f5_kept_by_survival.csv
